In [2]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [3]:
# Paths

DATA_DIR = Path("../data/blogfeedback")
DATA_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_DIR / "blogfeedback.zip"

DATA_URL = (
    "https://archive.ics.uci.edu/static/public/304/blogfeedback.zip"
)

In [4]:
# Download and extract the BlogFeedback dataset

if not ZIP_PATH.exists():
    print("Downloading BlogFeedback dataset...")
    urlretrieve(DATA_URL, ZIP_PATH)
    print("Download completed.")
else:
    print("Dataset archive already downloaded.")

with ZipFile(ZIP_PATH, "r") as zip_file:
    zip_file.extractall(DATA_DIR)

print("Extraction completed.")

Dataset archive already downloaded.
Extraction completed.


In [5]:
files = sorted(DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(files))

for file in files[:10]:
    print(file.name)

Number of CSV files: 61
blogData_test-2012.02.01.00_00.csv
blogData_test-2012.02.02.00_00.csv
blogData_test-2012.02.03.00_00.csv
blogData_test-2012.02.04.00_00.csv
blogData_test-2012.02.05.00_00.csv
blogData_test-2012.02.06.00_00.csv
blogData_test-2012.02.07.00_00.csv
blogData_test-2012.02.08.00_00.csv
blogData_test-2012.02.09.00_00.csv
blogData_test-2012.02.10.00_00.csv


In [6]:
TRAIN_FILE = DATA_DIR / "blogData_train.csv"

test_files = sorted(
    DATA_DIR.glob("blogData_test-*.csv")
)

print("Training file:", TRAIN_FILE.name)
print("Training file exists:", TRAIN_FILE.exists())

print("Number of test files:", len(test_files))
print("First test file:", test_files[0].name)
print("Last test file:", test_files[-1].name)

Training file: blogData_train.csv
Training file exists: True
Number of test files: 60
First test file: blogData_test-2012.02.01.00_00.csv
Last test file: blogData_test-2012.03.31.01_00.csv


In [7]:
# Load the real-world training data

train_data = np.loadtxt(
    TRAIN_FILE,
    delimiter=",",
    dtype=np.float64
)

print("Full training data shape:", train_data.shape)

Full training data shape: (52397, 281)


In [8]:
X_full_real = train_data[:, :-1]
y_full_real = train_data[:, -1]

print("X shape:", X_full_real.shape)
print("y shape:", y_full_real.shape)

X shape: (52397, 280)
y shape: (52397,)


In [9]:
# Load the latest temporally separated test set

TEST_FILE = test_files[-1]

test_data = np.loadtxt(
    TEST_FILE,
    delimiter=",",
    dtype=np.float64
)

X_test_real_full = test_data[:, :-1]
y_test_real = test_data[:, -1]

print("Test file:", TEST_FILE.name)
print("Test data shape:", test_data.shape)
print("X_test shape:", X_test_real_full.shape)
print("y_test shape:", y_test_real.shape)

Test file: blogData_test-2012.03.31.01_00.csv
Test data shape: (214, 281)
X_test shape: (214, 280)
y_test shape: (214,)


In [10]:
print("Training target:")
print("  min:", np.min(y_full_real))
print("  median:", np.median(y_full_real))
print("  mean:", np.mean(y_full_real))
print("  max:", np.max(y_full_real))

print("\nTest target:")
print("  min:", np.min(y_test_real))
print("  median:", np.median(y_test_real))
print("  mean:", np.mean(y_test_real))
print("  max:", np.max(y_test_real))

Training target:
  min: 0.0
  median: 0.0
  mean: 6.764719354161498
  max: 1424.0

Test target:
  min: 0.0
  median: 0.0
  mean: 7.08411214953271
  max: 632.0


In [11]:
# Real-world experiment settings

N_TRAIN_REAL = 150
D_MAX_REAL = X_full_real.shape[1]   # 280

real_rng = np.random.default_rng(RANDOM_SEED)

# Randomly select 150 observations from the official training data
train_indices = real_rng.choice(
    X_full_real.shape[0],
    size=N_TRAIN_REAL,
    replace=False
)

X_train_real_raw = X_full_real[train_indices]
y_train_real_raw = y_full_real[train_indices]

# Log-transform the highly skewed target
y_train_real = np.log1p(y_train_real_raw)
y_test_real_log = np.log1p(y_test_real)

print("Training subset shape:", X_train_real_raw.shape)
print("Transformed y train shape:", y_train_real.shape)

print("\nOriginal target mean:", np.mean(y_train_real_raw))
print("Log-target mean:", np.mean(y_train_real))

Training subset shape: (150, 280)
Transformed y train shape: (150,)

Original target mean: 5.533333333333333
Log-target mean: 0.6197627027871994


In [12]:
# Fixed random ordering of the 280 features

feature_order = real_rng.permutation(D_MAX_REAL)

X_train_real_ordered = X_train_real_raw[:, feature_order]
X_test_real_ordered = X_test_real_full[:, feature_order]

print("First 10 feature indices:", feature_order[:10])

First 10 feature indices: [186  40 254 159 213  51 214 151 100  91]


In [13]:
# Standardization using training statistics only

feature_mean = X_train_real_ordered.mean(axis=0)
feature_std = X_train_real_ordered.std(axis=0)

# Detect constant features
zero_variance = feature_std == 0

print("Zero-variance features:", np.sum(zero_variance))

# Avoid division by zero
feature_std_safe = feature_std.copy()
feature_std_safe[zero_variance] = 1.0

X_train_real_scaled = (
    X_train_real_ordered - feature_mean
) / feature_std_safe

X_test_real_scaled = (
    X_test_real_ordered - feature_mean
) / feature_std_safe

print("Scaled train shape:", X_train_real_scaled.shape)
print("Scaled test shape:", X_test_real_scaled.shape)

Zero-variance features: 108
Scaled train shape: (150, 280)
Scaled test shape: (214, 280)
